# Neural Network Volatility Model

This notebook is focused on training a multilayer perceptron nerual network to predict future 20-day realized volatility.

The model uses teh same selected features and time-based train/test split as all the other models so we can compare it fairly.

The neural network provides a different nonlinear modeling approach from the
linear, tree ensemble, and statistical time-series models already evaluated.

# Imports

In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Set Paths

In [2]:
FEATURES_PATH = Path("../../data/processed/features")
MODELING_PATH = Path("../../data/processed/modeling")
MODEL_OUTPUT_PATH = MODELING_PATH / "neural_network_mlp"

MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Output directory:", MODEL_OUTPUT_PATH)

Output directory: ..\..\data\processed\modeling\neural_network_mlp


# Load the Feature Dataset

In [3]:
df = pd.read_csv(
    FEATURES_PATH / "feature_engineered_dataset.csv",
    parse_dates=["Date"],
)

In [4]:
df = df.sort_values(['Date', 'ticker']).reset_index(drop=True)

## Shape and Info of Dataset

In [5]:
print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Ticker count:", df["ticker"].nunique())

Dataset shape: (41370, 30)
Date range: 2018-01-31 00:00:00 to 2025-12-01 00:00:00
Ticker count: 21


In [6]:
df.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_decimal,vix,treasury_10yr_pct,yield_curve_spread,is_inverted,fed_funds_rate_pct,...,rolling_return_20d,abs_return,squared_return,rolling_abs_return_20d,rolling_squared_return_20d,rolling_volatility_5d,rolling_volatility_20d,moving_avg_20d,price_to_moving_avg_20d,future_volatility_20d
0,2018-01-31,AAPL,39.138020,0.002755,0.0146,13.54,2.72,1.26,0,1.41,...,-0.001378,0.002755,7.589440e-06,0.006855,0.000087,0.011013,0.009462,40.695438,0.961730,0.022707
1,2018-01-31,AGG,84.439857,0.000833,0.0146,13.54,2.72,1.26,0,1.41,...,-0.000491,0.000833,6.946851e-07,0.001156,0.000002,0.001981,0.001411,84.804251,0.995703,0.002252
2,2018-01-31,AMZN,72.544502,0.009090,0.0146,13.54,2.72,1.26,0,1.41,...,0.010048,0.009090,8.263170e-05,0.011328,0.000193,0.003308,0.009819,65.750550,1.103329,0.023577
3,2018-01-31,CAT,136.724533,-0.005984,0.0146,13.54,2.72,1.26,0,1.41,...,0.002100,0.005984,3.580998e-05,0.009638,0.000147,0.014251,0.012264,139.401691,0.980795,0.026032
4,2018-01-31,GLD,127.650002,0.006703,0.0146,13.54,2.72,1.26,0,1.41,...,0.001005,0.006703,4.493635e-05,0.004544,0.000031,0.005561,0.005660,126.463501,1.009382,0.007289


# Load the Selected Features